In [ ]:
DATA_DIR = "/content/drive/MyDrive/pcam"

In [ ]:
!pip install h5py timm torch torchvision

In [ ]:
import h5py

def load_subset(x_path, y_path, limit=5000):
    with h5py.File(x_path, 'r') as f:
        X = f['x'][:limit]

    with h5py.File(y_path, 'r') as f:
        y = f['y'][:limit]

    return X, y

In [ ]:
DATA_DIR = "/content/drive/MyDrive/pcam"

X, y = load_subset(
    f"{DATA_DIR}/camelyonpatch_level_2_split_train_x.h5",
    f"{DATA_DIR}/camelyonpatch_level_2_split_train_y.h5",
    limit=5000   # to allow computation
)

print(X.shape, y.shape)

In [ ]:
import h5py

f = h5py.File(f"{DATA_DIR}/camelyonpatch_level_2_split_train_x.h5", 'r')
print(list(f.keys()))

In [ ]:
print(X.shape)

PREPROCESSING


In [ ]:
import cv2
import numpy as np

def preprocess(X):
    out = []
    for img in X:
        img = cv2.resize(img, (128, 128))
        img = img / 255.0
        out.append(img)
    return np.array(out)

X = preprocess(X)
print(X.shape)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class PCamDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = torch.tensor(self.X[idx], dtype=torch.float32).permute(2, 0, 1)
        label = torch.tensor(self.y[idx], dtype=torch.long)
        return img, label

dataset = PCamDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [11]:
import timm
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

model = timm.create_model('resnet18', pretrained=True)
model.fc = nn.Identity()
model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act2): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, m

In [12]:
features = []
labels = []

with torch.no_grad():
    for imgs, y_batch in loader:
        imgs = imgs.to(device)

        feats = model(imgs)

        features.append(feats.cpu())
        labels.append(y_batch)

features = torch.cat(features)
labels = torch.cat(labels)

print(features.shape)

torch.Size([5000, 512])


In [13]:
from sklearn.neighbors import NearestNeighbors

def build_graph(features, k=8):
    nbrs = NearestNeighbors(n_neighbors=k, metric='euclidean')
    nbrs.fit(features)

    distances, indices = nbrs.kneighbors(features)

    edge_index = []

    for i in range(len(indices)):
        for j in indices[i]:
            edge_index.append([i, j])

    return edge_index

In [14]:
edge_index = build_graph(features.numpy(), k=8)
print(len(edge_index))

40000


In [15]:
!pip install torch-geometric

In [16]:
import torch
from torch_geometric.data import Data

edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

x = features
y = labels

data = Data(x=x, edge_index=edge_index, y=y)

print(data)

Data(x=[5000, 512], edge_index=[2, 40000], y=[5000, 1, 1, 1])


In [17]:
import networkx as nx

def compute_pagerank(edge_index):
    G = nx.Graph()

    edges = edge_index.t().tolist()
    G.add_edges_from(edges)

    pr = nx.pagerank(G, alpha=0.85)

    scores = [pr[i] for i in range(len(pr))]
    return torch.tensor(scores, dtype=torch.float32)

In [18]:
pagerank_scores = compute_pagerank(edge_index)
print(pagerank_scores.shape)

torch.Size([5000])


In [19]:
pagerank_scores = pagerank_scores.unsqueeze(1)

x_aug = torch.cat([x, pagerank_scores], dim=1)

print(x_aug.shape)

torch.Size([5000, 513])


In [20]:
from torch_geometric.nn import GCNConv
import torch.nn.functional as F

class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(513, 128)
        self.conv2 = GCNConv(128, 2)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

In [ ]:
model = GCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

data = data.to(device)
data.x = x_aug.to(device)

data.y = data.y.squeeze()

for epoch in range(20):
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index)

    loss = F.cross_entropy(out, data.y)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 0.7353984713554382
Epoch 1, Loss: 4.3220391273498535
Epoch 2, Loss: 0.6490403413772583
Epoch 3, Loss: 1.933984398841858
Epoch 4, Loss: 1.402522325515747
Epoch 5, Loss: 0.6775600910186768
Epoch 6, Loss: 0.6396998763084412
Epoch 7, Loss: 0.8109577298164368
Epoch 8, Loss: 0.8110615611076355
Epoch 9, Loss: 0.6873934268951416
Epoch 10, Loss: 0.5759150981903076
Epoch 11, Loss: 0.5693682432174683
Epoch 12, Loss: 0.6276790499687195
Epoch 13, Loss: 0.6316230297088623
Epoch 14, Loss: 0.5993127226829529
Epoch 15, Loss: 0.567411482334137
Epoch 16, Loss: 0.5462108254432678
Epoch 17, Loss: 0.5390164256095886
Epoch 18, Loss: 0.5528687834739685
Epoch 19, Loss: 0.5548933744430542


In [23]:
model.eval()

with torch.no_grad():
    out = model(data.x, data.edge_index)
    preds = out.argmax(dim=1)

In [25]:
y_true = data.y.cpu().numpy()
y_pred = preds.cpu().numpy()

In [26]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)

Accuracy: 0.7406
Precision: 0.6851303317535545
Recall: 0.9081272084805654
F1 Score: 0.7810231301705217


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class PRGAT(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.gat1 = GATConv(in_channels, 128, heads=4, concat=True)
        self.gat2 = GATConv(128*4, 2, heads=1, concat=False)

    def forward(self, x, edge_index, pagerank):
        x = x * pagerank.unsqueeze(1)

        x = self.gat1(x, edge_index)
        x = F.relu(x)

        x = self.gat2(x, edge_index)

        return x

In [ ]:
model = PRGAT(x.shape[1]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

data = data.to(device)
pagerank_scores = pagerank_scores.to(device)

data.y = data.y.squeeze()

for epoch in range(20):
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index, pagerank_scores)

    loss = F.cross_entropy(out, data.y)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item()}")